# Metal vs Non-metal Classification (SVM)

This notebook uses synthetic range–Doppler heatmaps to train and evaluate an SVM classifier.

- Dataset: 400 samples for train/validation and a held-out 100-sample test set
- Split: 80/20 train/val from the 400-sample pool (320 train, 80 val)
- Model: SVM (RBF) with StandardScaler + PCA (retain 95% variance)
- Metrics: Accuracy, confusion matrix, ROC AUC, PR AP for both val and test
- Artifacts saved under `models/`: model `.joblib` + PNG plots

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from radar_simulation import generate_dataset

# Paths
project_root = os.path.abspath('.')
data_dir = os.path.join(project_root, 'data')
trainval_dir = os.path.join(data_dir, 'trainval')
test_dir = os.path.join(data_dir, 'test')
models_dir = os.path.join(project_root, 'models')
os.makedirs(models_dir, exist_ok=True)

# Ensure datasets exist (400 train/val + 100 test)
trainval_csv = os.path.join(trainval_dir, 'labels.csv')
test_csv = os.path.join(test_dir, 'labels.csv')
if not os.path.exists(trainval_csv):
    os.makedirs(trainval_dir, exist_ok=True)
    generate_dataset(trainval_dir, n_samples=400, img_size=(64, 64))
if not os.path.exists(test_csv):
    os.makedirs(test_dir, exist_ok=True)
    generate_dataset(test_dir, n_samples=100, img_size=(64, 64))
print('Train/Val CSV:', trainval_csv)
print('Test CSV:', test_csv)

# Loader helper
import csv

def load_from_csv(csv_path):
    paths = []
    labels = []
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for r in reader:
            paths.append(r['npy_path'])
            labels.append(int(r['label']))
    X = np.stack([np.load(p) for p in paths]).astype('float32')
    y = np.array(labels)
    return X, y

# Load train/val pool and split
X_trainval, y_trainval = load_from_csv(trainval_csv)
X_trainval = X_trainval[..., None]
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval
)
print('Shapes -> train:', X_train.shape, 'val:', X_val.shape)

# Load independent test set
X_test, y_test = load_from_csv(test_csv)
X_test = X_test[..., None].astype('float32')
print('Test shape ->', X_test.shape)


Train/Val CSV: c:\Users\kalya\Downloads\AIR Guruji Assignment\radar_project\data\trainval\labels.csv
Test CSV: c:\Users\kalya\Downloads\AIR Guruji Assignment\radar_project\data\test\labels.csv
Shapes -> train: (320, 64, 64, 1) val: (80, 64, 64, 1)
Shapes -> train: (320, 64, 64, 1) val: (80, 64, 64, 1)
Test shape -> (100, 64, 64, 1)
Test shape -> (100, 64, 64, 1)
CNN training skipped due to TensorFlow issue: Traceback (most recent call last):
  File "c:\Users\kalya\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/iss

In [8]:
# SVM classifier with separate validation and test evaluations
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
import matplotlib.pyplot as plt
import numpy as np
import os

# Flatten for classical model
X_train_flat = X_train.reshape((X_train.shape[0], -1))
X_val_flat = X_val.reshape((X_val.shape[0], -1))
X_test_flat = X_test.reshape((X_test.shape[0], -1))

svm_clf = Pipeline([
    ('scaler', StandardScaler(with_mean=True)),
    ('pca', PCA(n_components=0.95, svd_solver='full')),
    ('svc', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42))
])

svm_clf.fit(X_train_flat, y_train)

# Validation metrics
svm_pred_val = svm_clf.predict(X_val_flat)
svm_prob_val = svm_clf.predict_proba(X_val_flat)[:, 1]
acc_val = accuracy_score(y_val, svm_pred_val)
cm_val = confusion_matrix(y_val, svm_pred_val)
prec_val, rec_val, _ = precision_recall_curve(y_val, svm_prob_val)
ap_val = average_precision_score(y_val, svm_prob_val)
fpr_val, tpr_val, _ = roc_curve(y_val, svm_prob_val)
roc_auc_val = auc(fpr_val, tpr_val)

print(f'Validation Accuracy (SVM): {acc_val:.3f}')
print('SVM Validation Confusion matrix:\n', cm_val)
print('SVM Validation Report:\n', classification_report(y_val, svm_pred_val))

# Test metrics
svm_pred_test = svm_clf.predict(X_test_flat)
svm_prob_test = svm_clf.predict_proba(X_test_flat)[:, 1]
acc_test = accuracy_score(y_test, svm_pred_test)
cm_test = confusion_matrix(y_test, svm_pred_test)
prec_test, rec_test, _ = precision_recall_curve(y_test, svm_prob_test)
ap_test = average_precision_score(y_test, svm_prob_test)
fpr_test, tpr_test, _ = roc_curve(y_test, svm_prob_test)
roc_auc_test = auc(fpr_test, tpr_test)

print(f'Test Accuracy (SVM): {acc_test:.3f}')
print('SVM Test Confusion matrix:\n', cm_test)
print('SVM Test Report:\n', classification_report(y_test, svm_pred_test))

# Save model
import joblib
os.makedirs(models_dir, exist_ok=True)
svm_path = os.path.join(models_dir, 'metal_classifier_svm.joblib')
joblib.dump(svm_clf, svm_path)
print('Saved SVM model to', svm_path)

# Plot helpers

def save_confusion(cm, title, filename):
    plt.figure(figsize=(4, 4))
    plt.imshow(cm, cmap='Blues', interpolation='nearest')
    plt.title(title)
    plt.colorbar()
    plt.xticks([0, 1], ['Non-metal', 'Metal'])
    plt.yticks([0, 1], ['Non-metal', 'Metal'])
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha='center', va='center', color='black')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.savefig(os.path.join(models_dir, filename), dpi=150, bbox_inches='tight')
    plt.close()


def save_roc(fpr, tpr, auc_val, title, filename):
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f'AUC = {auc_val:.3f}')
    plt.plot([0, 1], [0, 1], 'k--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.savefig(os.path.join(models_dir, filename), dpi=150, bbox_inches='tight')
    plt.close()


def save_pr(rec, prec, ap_val, title, filename):
    plt.figure(figsize=(5, 4))
    plt.plot(rec, prec, label=f'AP = {ap_val:.3f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.legend(loc='lower left')
    plt.savefig(os.path.join(models_dir, filename), dpi=150, bbox_inches='tight')
    plt.close()

# Save validation plots
save_confusion(cm_val, 'Confusion Matrix (SVM Val)', 'confusion_matrix_svm_val.png')
save_roc(fpr_val, tpr_val, roc_auc_val, 'ROC Curve (SVM Val)', 'roc_curve_svm_val.png')
save_pr(rec_val, prec_val, ap_val, 'PR Curve (SVM Val)', 'pr_curve_svm_val.png')

# Save test plots
save_confusion(cm_test, 'Confusion Matrix (SVM Test)', 'confusion_matrix_svm_test.png')
save_roc(fpr_test, tpr_test, roc_auc_test, 'ROC Curve (SVM Test)', 'roc_curve_svm_test.png')
save_pr(rec_test, prec_test, ap_test, 'PR Curve (SVM Test)', 'pr_curve_svm_test.png')


Validation Accuracy (SVM): 0.900
SVM Validation Confusion matrix:
 [[35  5]
 [ 3 37]]
SVM Validation Report:
               precision    recall  f1-score   support

           0       0.92      0.88      0.90        40
           1       0.88      0.93      0.90        40

    accuracy                           0.90        80
   macro avg       0.90      0.90      0.90        80
weighted avg       0.90      0.90      0.90        80

Test Accuracy (SVM): 0.910
SVM Test Confusion matrix:
 [[44  6]
 [ 3 47]]
SVM Test Report:
               precision    recall  f1-score   support

           0       0.94      0.88      0.91        50
           1       0.89      0.94      0.91        50

    accuracy                           0.91       100
   macro avg       0.91      0.91      0.91       100
weighted avg       0.91      0.91      0.91       100

Saved SVM model to c:\Users\kalya\Downloads\AIR Guruji Assignment\radar_project\models\metal_classifier_svm.joblib
